In [7]:
import pandas as pd
import time
# Load the CSV file
df = pd.read_csv("IOT Data Simulation/smart_logistic_tracker_japan.csv")

# Display the first 5 rows
df.head()

,timestamp,carrier,tracking_number,package_id,origin,current_location,delivery_location,prefecture,latitude,longitude,...,waiting_time_minutes,perishable,temperature,humidity,rfid_tag,rfid_verified,tamper_alert,traffic_status,inventory_level,asset_utilization
0,2026-05-04 13:50:26.857905,Yamato Transport,942646961460,PKG7545,Tokyo,Naha Central Post Office,Tokyo,Kanagawa,35.993159,139.038781,...,45,No,10.7,40,RFID736892,False,No,Heavy,99,84.59
1,2026-05-03 23:36:26.858095,Japan Post,74355111775,PKG2659,Tokyo,Nagoya Central Post Office,Kyoto,Kanagawa,35.691292,139.130870,...,54,Yes,6.2,82,RFID156229,False,Yes,Detour,363,53.39
2,2026-05-04 08:32:26.858217,Japan Post,217497030475,PKG7965,Osaka,Nagoya Central Post Office,Osaka,Aichi,35.591109,139.784940,...,144,Yes,-3.1,86,RFID890703,True,Yes,Heavy,25,95.75
3,2026-05-04 02:35:26.858332,Japan Post,249781996688,PKG5296,Fukuoka,Sapporo Central Post Office,Sapporo,Osaka,35.570440,139.689163,...,173,Yes,6.3,87,RFID603182,True,Yes,Detour,145,63.84
4,2026-05-04 08:28:26.858444,Japan Post,718415724062,PKG9987,Fukuoka,Yokohama Sales Office,Sapporo,Hokkaido,35.679375,139.408071,...,82,No,0.7,60,RFID921432,False,Yes,Detour,34,56.28


In [8]:
from web3 import Web3

# Connect to local blockchain
ganache_url = "http://127.0.0.1:8545"
web3 = Web3(Web3.HTTPProvider(ganache_url))

# Verify connection
if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

✅ Connected to Ganache successfully!


In [9]:
import json

# Replace with actual contract address from Remix
contract_address = "0xF633071fB31C49Fd5C805Cfd16e889F5F2952a67"

# Load the ABI that matches the deployed contract in this repository
with open("abi.json", "r", encoding="utf-8") as abi_file:
    abi = json.load(abi_file)

# Load the smart contract
contract = web3.eth.contract(address=contract_address, abi=abi)

# Use the contract owner as the transaction sender so onlyOwner checks pass
contract_owner = contract.functions.owner().call()
if contract_owner in web3.eth.accounts:
    web3.eth.default_account = contract_owner
else:
    web3.eth.default_account = web3.eth.accounts[0]
    print(f"⚠️ Contract owner {contract_owner} is not unlocked in Ganache; using {web3.eth.default_account} instead.")

print(f"✅ Connected to Smart Contract at {contract_address}")
print(f"✅ Using sender account: {web3.eth.default_account}")

✅ Connected to Smart Contract at 0xF633071fB31C49Fd5C805Cfd16e889F5F2952a67
✅ Using sender account: 0x2c538276DE805CB27b3b41183FA731f65F319Fa1


In [10]:
def send_iot_data(package_id, data_type, data_value):
    """
    Sends logistics IoT data
    to the deployed smart contract
    """

    txn = contract.functions.storeData(
        package_id,
        data_type,
        data_value
    ).transact({
        'from': web3.eth.default_account,
        'gas': 3000000
    })

    # Wait for transaction confirmation
    receipt = web3.eth.wait_for_transaction_receipt(txn)

    print(
        f"✅ Data Stored | {package_id} | "
        f"Type: {data_type} | "
        f"Value: {data_value} | "
        f"Txn Hash: {receipt.transactionHash.hex()}"
    )

# Store 100 records
for index, row in df.head(100).iterrows():
    package_id = str(row["package_id"])
    location = str(row["current_location"])
    status = str(row["latest_status"])

    send_iot_data(package_id, "Location", location)
    send_iot_data(package_id, "Status", status)

    # Delay between transactions
    time.sleep(0.1)

print("\n✅ Successfully stored 100 records on the blockchain!")

✅ Data Stored | PKG7545 | Type: Location | Value: Naha Central Post Office | Txn Hash: 61ae3470ff32938eeae17ac6ee13d35c9169058ad7787448ea41a67686814c29
✅ Data Stored | PKG7545 | Type: Status | Value: Out for Delivery | Txn Hash: 4231d2243591ea3c338849fc8323cbeb02e232a41b9970f7a77ae8d947c21903
✅ Data Stored | PKG2659 | Type: Location | Value: Nagoya Central Post Office | Txn Hash: b1b3e29447d08e74c9050a819e96354b3ba1b77667a3cb3e8e35920598c15aa1
✅ Data Stored | PKG2659 | Type: Status | Value: Arrival | Txn Hash: e27afd0583ce541d5c6cd84c5ce4fbd053dd596659eb1238143945666381f2fe
✅ Data Stored | PKG7965 | Type: Location | Value: Nagoya Central Post Office | Txn Hash: 1ece20663eeb704d77a9f293f6a58f2886cbca0be9ce8dbb379f54723f6e9a31
✅ Data Stored | PKG7965 | Type: Status | Value: Storage | Txn Hash: d0799de5cc64416aad4d88ce06c3373d54b1dbeb0a5a2ce57781c84b7d5fcc6a
✅ Data Stored | PKG5296 | Type: Location | Value: Sapporo Central Post Office | Txn Hash: 78efea1a1b39f5b34f6eb704deac50dd6e0c859be9

In [11]:
current_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {current_records}")

Total IoT records stored: 400


In [12]:
# Retrieve and display the first stored record
first_record = contract.functions.getRecord(0).call()

print("📦 First Stored Record")
print(f"Timestamp: {first_record[0]}")
print(f"Package ID: {first_record[1]}")
print(f"Data Type: {first_record[2]}")
print(f"Data Value: {first_record[3]}")

📦 First Stored Record
Timestamp: 1780191476
Package ID: PKG7545
Data Type: Location
Data Value: Naha Central Post Office
